In [1]:
# import required Libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

print("TF:", tf.__version__)
print("PD:", pd.__version__)

TF: 2.19.0
PD: 2.2.2


In [2]:
# step 1: checking if gpu available or not. If available allocating gpu
# Note after that tensorflow keras will automatically run all operation on the GPU
# we don't need make any changes in the further code for explicitly mention to gpu

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU available, memory growth set.")
else:
    print("No GPU detected, running on CPU.")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU available, memory growth set.


In [3]:
# Step 2: Loading telco.csv (attached dataset)

csv_path = "telco.csv"   # making sure this matches the uploaded filename

df = pd.read_csv(csv_path)

df.shape        # (rows, columns)

(7043, 50)

In [4]:
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [5]:
print("\nInfo: (null values, data types)")
df.info()


Info: (null values, data types)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Under 30                           7043 non-null   object 
 4   Senior Citizen                     7043 non-null   object 
 5   Married                            7043 non-null   object 
 6   Dependents                         7043 non-null   object 
 7   Number of Dependents               7043 non-null   int64  
 8   Country                            7043 non-null   object 
 9   State                              7043 non-null   object 
 10  City                               7043 non-null   object 
 11  Zip Code               

In [6]:
print("\nChurn Label value counts:")
df["Churn Label"].value_counts()


Churn Label value counts:


,count
Churn Label,
No,5174
Yes,1869


In [7]:
# Step 3: Creating numeric target column (churn from churn label) and drop ID + Churn label

target_col = "Churn"  # name for our numeric target column

# 1) Create numeric 'Churn' from text 'Churn Label' ("Yes"/"No" -> 1/0)
# we need to convert strings to integers for the model.
df[target_col] = df["Churn Label"].map({"Yes": 1, "No": 0}).astype("int32")

# 2) Drop 'Customer ID' (identifier, not a useful feature) AND 'Churn Label' (text version of target)
# we now explicitly drop 'Churn Label' so that it is NOT used as a feature.
df = df.drop(columns=["Customer ID", "Churn Label"])

In [8]:
# Step 4: Identify numeric and categorical columns

# Numeric columns: int64/float64, EXCLUDING the target
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]

# Categorical columns: object (string) columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("\nCategorical columns:", categorical_cols)

Numeric columns: ['Age', 'Number of Dependents', 'Zip Code', 'Latitude', 'Longitude', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Satisfaction Score', 'Churn Score', 'CLTV']

Categorical columns: ['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Country', 'State', 'City', 'Quarter', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Customer Status', 'Churn Category', 'Churn Reason']


In [9]:
# Step 5: Handle missing values BEFORE encoding
# ----------------------------------------------------------------
# Numeric columns:
#   - fill NaN with the column median (simple, robust choice).
# Categorical columns:
#   - fill NaN with a special category "Unknown" so get_dummies can handle it.

for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [10]:
# Step 6: One-hot encode categorical features
#   - get_dummies turns each category into separate 0/1 columns.

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Encoded shape:", df_encoded.shape)

Encoded shape: (7043, 1182)


In [11]:
# Step 7: Split into features X and target y
X = df_encoded.drop(columns=[target_col]).values.astype("float32")
y = df_encoded[target_col].values.astype("int32")

print("X shape:", X.shape)  # (num_samples, num_features_after_encoding)
print("y shape:", y.shape)  # (num_samples,)

X shape: (7043, 1181)
y shape: (7043,)


In [12]:
from sklearn.model_selection import train_test_split

# Step 8: Train/val/test split
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keep churn proportion similar in temp/test sets
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2,
    random_state=42,
    stratify=y_temp  # keep churn proportion similar in train/validation sets
)

print("Train shape:", X_train_sub.shape, y_train_sub.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (4507, 1181) (4507,)
Validation shape: (1127, 1181) (1127,)
Test shape: (1409, 1181) (1409,)


In [13]:
from sklearn.preprocessing import StandardScaler

# Step 9: Scale features with StandardScaler (fit on train, apply to train + test)
scaler = StandardScaler()
X_train_sub_scaled = scaler.fit_transform(X_train_sub)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled train mean (first 5 features):", X_train_sub_scaled.mean(axis=0)[:5])
print("Scaled train std  (first 5 features):", X_train_sub_scaled.std(axis=0)[:5])

Scaled train mean (first 5 features): [ 8.3184650e-09 -5.6999337e-08  1.6319532e-08  1.7985871e-08
 -2.3222933e-08]
Scaled train std  (first 5 features): [1.000002  1.0000093 1.0000001 0.9999995 0.9999989]


In [14]:
# Step 10: Create tf.data Datasets for train, validation, and test
# Build tf.data.Dataset objects from tensors

batch_size = 64

# Training dataset:
train_ds = tf.data.Dataset.from_tensor_slices((X_train_sub_scaled, y_train_sub))

# Shuffle: shuffle the order of samples each epoch (buffer_size ~ number of samples)
train_ds = train_ds.shuffle(buffer_size=len(X_train_sub_scaled))

# Batch: group samples into mini-batches of 'batch_size'
train_ds = train_ds.batch(batch_size)

# Prefetch: overlap data preparation and model execution for speed
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

# Validation dataset:
val_ds = tf.data.Dataset.from_tensor_slices((X_val_scaled, y_val))
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Test dataset:
test_ds = tf.data.Dataset.from_tensor_slices((X_test_scaled, y_test))
test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [15]:
from tensorflow.keras import regularizers

# Step 11: Define a lambda value (for L2 regularization) and dropout rate

reg_lambda = 0.001  # L2 regularization strength (hyperparameter you can tune)
drop_rate = 0.3     # Dropout rate (fraction of units to drop)

**Creating model with the functional API**

In [16]:
# step 12: Build the Keras model for churn prediction with dropout and L2 regularization

input_dim = X_train_sub_scaled.shape[1]  # number of input features after encoding
inputs = tf.keras.Input(shape=(input_dim,))

# Dense layer with L2 regularization on weights (kernel_regularizer)
x = layers.Dense(
    128,
    activation="relu",
    kernel_regularizer=regularizers.l2(reg_lambda)  # L2 penalty on large weights
    )(inputs)

# Dropout layer
#  - During training, randomly sets 30% of the outputs of the previous layer to 0.
#  - This forces the network not to rely too heavily on any single neuron; reduces overfitting.
x = layers.Dropout(drop_rate)(x)

x = layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=regularizers.l2(reg_lambda)  # L2 again
    )(x)

x = layers.Dropout(drop_rate)(x)  # Dropout after another Dense layer

x = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(reg_lambda)
    )(x)

x = layers.Dropout(drop_rate)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

reg_churn_model = tf.keras.Model(inputs=inputs, outputs=outputs)

reg_churn_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1181)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       151,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 161,665 (631.50 KB)

 Trainable params: 161,665 (631.50 KB)

 Non-trainable params: 0 (0.00 B)

**Creating model using Sequential Model(Container)**

In [ ]:
input_dim = X_train_sub_scaled.shape[1]

reg_churn_model = tf.keras.Sequential([

    layers.Input(shape=(input_dim, )),

    layers.Dense(128,
                 activation="relu",
                 kernel_regularizer=regularizers.l2(reg_lambda)
                 ),

    layers.Dropout(drop_rate),

    layers.Dense(64,
                 activation="relu",
                 kernel_regularizer=regularizers.l2(reg_lambda)
                 ),

    layers.Dropout(drop_rate),

    layers.Dense(32,
                 activation="relu",
                 kernel_regularizer=regularizers.l2(reg_lambda)
                 ),

    layers.Dropout(drop_rate),

    layers.Dense(1, activation="sigmoid")
])

reg_churn_model.summary()

In [17]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Step 13: Compile and train the model with callbacks

reg_churn_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"]
)

early_stop_reg = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint_reg = ModelCheckpoint(
    "best_reg_churn_tfdata.keras",
    monitor="val_loss",
    save_best_only=True
)

# Train the model using tf.data datasets

history_reg_ds = reg_churn_model.fit(
    train_ds,                 # (X_train_sub_scaled, y_train_sub)
    epochs=60,
    validation_data=val_ds,   # (X_val_scaled, y_val)
    callbacks=[early_stop_reg, checkpoint_reg],
    verbose=2
)

Epoch 1/60
71/71 - 10s - 138ms/step - accuracy: 0.8178 - loss: 0.6838 - val_accuracy: 0.9823 - val_loss: 0.3049
Epoch 2/60
71/71 - 0s - 4ms/step - accuracy: 0.9905 - loss: 0.2591 - val_accuracy: 0.9973 - val_loss: 0.2063
Epoch 3/60
71/71 - 0s - 4ms/step - accuracy: 0.9989 - loss: 0.1932 - val_accuracy: 0.9991 - val_loss: 0.1761
Epoch 4/60
71/71 - 0s - 4ms/step - accuracy: 0.9993 - loss: 0.1686 - val_accuracy: 1.0000 - val_loss: 0.1553
Epoch 5/60
71/71 - 0s - 4ms/step - accuracy: 1.0000 - loss: 0.1474 - val_accuracy: 0.9991 - val_loss: 0.1381
Epoch 6/60
71/71 - 0s - 4ms/step - accuracy: 0.9998 - loss: 0.1303 - val_accuracy: 1.0000 - val_loss: 0.1215
Epoch 7/60
71/71 - 0s - 4ms/step - accuracy: 1.0000 - loss: 0.1148 - val_accuracy: 1.0000 - val_loss: 0.1063
Epoch 8/60
71/71 - 0s - 4ms/step - accuracy: 1.0000 - loss: 0.1007 - val_accuracy: 1.0000 - val_loss: 0.0933
Epoch 9/60
71/71 - 0s - 4ms/step - accuracy: 1.0000 - loss: 0.0883 - val_accuracy: 1.0000 - val_loss: 0.0815
Epoch 10/60
71/7

In [18]:
print("Regularized - Last 5 training losses:", history_reg_ds.history["loss"][-5:])
print("Regularized - Last 5 validation losses:", history_reg_ds.history["val_loss"][-5:])
print("Regularized - Last 5 training accuracies:", history_reg_ds.history["accuracy"][-5:])
print("Regularized - Last 5 validation accuracies:", history_reg_ds.history["val_accuracy"][-5:])

Regularized - Last 5 training losses: [0.008565225638449192, 0.013384622521698475, 0.011072438210248947, 0.009943332523107529, 0.010167553089559078]
Regularized - Last 5 validation losses: [0.007585855666548014, 0.010501700453460217, 0.010322613641619682, 0.008696397766470909, 0.01723257452249527]
Regularized - Last 5 training accuracies: [0.999778151512146, 0.9988906383514404, 0.9995562434196472, 1.0, 0.999778151512146]
Regularized - Last 5 validation accuracies: [1.0, 1.0, 1.0, 1.0, 0.9991126656532288]


In [19]:
print("Best val_loss seen:", min(history_reg_ds.history["val_loss"]))
print("Epoch index of best val_loss:", np.argmin(history_reg_ds.history["val_loss"]))
print("Total epochs trained:", len(history_reg_ds.history["val_loss"]))

Best val_loss seen: 0.007512965705245733
Epoch index of best val_loss: 39
Total epochs trained: 45


In [20]:
# Step 14: Evaluate the model on the test dataset
# ---------------------------------------------
# test_ds is a tf.data.Dataset built from (X_test_scaled, y_test),
# batched and prefetched, just like train_ds and val_ds.

test_loss_ds, test_acc_ds = reg_churn_model.evaluate(test_ds, verbose=2)

print("Test loss (tf.data):", test_loss_ds)
print("Test accuracy (tf.data):", test_acc_ds)

23/23 - 1s - 28ms/step - accuracy: 0.9993 - loss: 0.0084
Test loss (tf.data): 0.008449164219200611
Test accuracy (tf.data): 0.9992902874946594


In [21]:
# Step 15: Predict churn for a manually specified customer
# ------------------------------------------------------

#  - The model was trained on df_encoded (after filling NaNs, one-hot, scaling, etc.).
#  - For a new customer, we must apply the SAME preprocessing steps and column order.
#    Means, pass it through the same pipeline.

# Create a single-row DataFrame starting from an existing row
# (this guarantees all columns exist, with correct names and types).
sample_original = df.iloc[[0]].copy()  # double brackets -> keep it as DataFrame (1 row)

# Now modify some values to simulate a new customer.
sample_original["Age"] = 30
sample_original["Tenure in Months"] = 5
sample_original["Monthly Charge"] = 80.0
sample_original["Contract"] = "Month-to-Month"
sample_original["Internet Type"] = "Fiber Optic"
sample_original["Offer"] = "Offer E"

print("Sample original row (before encoding):")
print(sample_original)

#     - Apply the SAME preprocessing as training:
#     - fill numeric NaNs with median
#     - fill categorical NaNs with "Unknown"
#     - one-hot encode with SAME columns as df_encoded
#     - scale with the SAME scaler (already fitted)

# a) Handle missing values for this single row
sample_numeric = sample_original[numeric_cols].copy()
sample_categorical = sample_original[categorical_cols].copy()

for col in numeric_cols:
    median_val = df[col].median()  # use TRAINING df median
    sample_numeric[col] = sample_numeric[col].fillna(median_val)

for col in categorical_cols:
    sample_categorical[col] = sample_categorical[col].fillna("Unknown")

# Recombine numeric + categorical for this row
sample_clean = pd.concat([sample_numeric, sample_categorical, sample_original[[target_col]]], axis=1)

# b) One-hot encode categoricals for this row
sample_encoded = pd.get_dummies(sample_clean, columns=categorical_cols, drop_first=True)

# c) Align columns with training df_encoded
#    - We reindex columns to match df_encoded's columns.
#    - Any missing column is filled with 0 (meaning that category is not present for this row).
#    - We then drop the target column to get features X.

sample_encoded_aligned = sample_encoded.reindex(columns=df_encoded.columns, fill_value=0)
sample_X = sample_encoded_aligned.drop(columns=[target_col]).values.astype("float32")

# d) Scale using the SAME scaler as training
sample_X_scaled = scaler.transform(sample_X)

# Predict with the trained model

sample_prob = reg_churn_model.predict(sample_X_scaled)[0, 0]  # single scalar probability
sample_pred_label = int(sample_prob >= 0.5)  # 0 or 1

print("\nPredicted churn probability:", float(sample_prob))
print("Predicted churn label (0 = No, 1 = Yes):", sample_pred_label)

if sample_pred_label == 1:
    print("Model prediction: This customer is LIKELY to churn (Yes).")
else:
    print("Model prediction: This customer is NOT likely to churn (No).")

Sample original row (before encoding):
  Gender  Age Under 30 Senior Citizen Married Dependents  \
0   Male   30       No            Yes      No         No   

   Number of Dependents        Country       State         City  ...  \
0                     0  United States  California  Los Angeles  ...   

   Total Extra Data Charges  Total Long Distance Charges  Total Revenue  \
0                        20                          0.0          59.65   

   Satisfaction Score Customer Status Churn Score  CLTV  Churn Category  \
0                   3         Churned          91  5433      Competitor   

                   Churn Reason Churn  
0  Competitor offered more data     1  

[1 rows x 49 columns]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 534ms/step

Predicted churn probability: 0.9989591836929321
Predicted churn label (0 = No, 1 = Yes): 1
Model prediction: This customer is LIKELY to churn (Yes).
